# Advanced Research Distillation (Distill-Research.ipynb)
- More sophisticated distillation approach
- Includes distributed training capabilities
- Uses Rotary Positional Embeddings (RoPE)
- Comprehensive configuration system
- Mixed precision training

In [ ]:
import os
import time
import math
import random
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.optim import AdamW
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.cuda.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint
from transformers import GPT2Tokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset, Dataset as HFDataset
from tqdm.auto import tqdm
from typing import Optional, Tuple, Dict, List, Any, Union


In [ ]:
# ========================================
# Configuration and Setup
# ========================================

class Config:
    """Training and model configuration"""
    # Model configuration
    vocab_size = 50257  # GPT-2 vocabulary size
    max_seq_len = 1024
    n_embd = 768  # Embedding dimension
    n_layer = 12  # Number of transformer layers
    n_head = 12  # Number of attention heads
    dropout_prob = 0.1
    
    # Training configuration
    batch_size = 16
    gradient_accumulation_steps = 2
    epochs = 5
    warmup_steps = 1000
    learning_rate = 5e-5
    weight_decay = 0.01
    
    # Advanced techniques
    use_alibi = True  # Use ALiBi positional embeddings
    use_flash_attention = True  # Use flash attention when available
    use_gradient_checkpointing = True  # Use gradient checkpointing to save memory
    use_mixed_precision = True  # Use mixed precision training
    use_mla = True  # Use Multi-head Latent Attention
    mla_n_latent = 16  # Number of latent tokens in MLA
    
    # Knowledge distillation
    use_distillation = True
    temperature = 2.0
    alpha = 0.5  # Weight for distillation loss vs task loss
    
    # Optimization & checkpointing
    checkpoint_interval = 600  # Save checkpoints every 10 minutes
    checkpoint_dir = './checkpoints'
    
    # Data configuration
    data_files = None  # Set to a list of files or use a dataset from Hugging Face
    dataset_name = "wikitext"
    dataset_config = "wikitext-103-v1"
    
    # Distributed training
    distributed = False
    world_size = torch.cuda.device_count() if torch.cuda.is_available() else 1
    
    def __init__(self, **kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)

def setup_environment(cfg, rank=0):
    """Setup training environment and reproducibility"""
    # Set up device
    if torch.cuda.is_available():
        device = torch.device(f'cuda:{rank}')
        torch.cuda.set_device(device)
    else:
        device = torch.device('cpu')
    
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    
    # Setup logging
    if rank == 0:  # Only log from main process
        os.makedirs('logs', exist_ok=True)
        logging.basicConfig(
            filename='logs/training_log.txt',
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
    
    return device

def setup_distributed(rank, world_size):
    """Initialize distributed training environment"""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl" if torch.cuda.is_available() else "gloo", 
                            rank=rank, 
                            world_size=world_size)

In [ ]:
# ========================================
# Advanced Model Architecture
# ========================================

class RMSNorm(nn.Module):
    """RMS Normalization layer"""
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        """Apply RMS normalization
        Args:
            x: Input tensor of shape [batch_size, seq_len, dim]
        Returns:
            Normalized tensor of the same shape
        """
        # Calculate RMS
        norm_x = torch.sqrt(torch.mean(x.pow(2), dim=-1, keepdim=True) + self.eps)
        # Normalize and scale
        x_normed = x / norm_x
        return x_normed * self.weight

class RotaryEmbedding(nn.Module):
    """Rotary positional embeddings"""
    def __init__(self, dim, base=10000):
        super().__init__()
        inv_freq = 1. / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self.seq_len_cached = None
        self.cos_cached = None
        self.sin_cached = None
    
    def forward(self, x, seq_len):
        """Get rotary embeddings
        Args:
            x: Input tensor
            seq_len: Sequence length
        Returns:
            Tuple of (cos, sin) for rotary embeddings
        """
        if seq_len != self.seq_len_cached:
            self.seq_len_cached = seq_len
            t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
            freqs = torch.einsum('i,j->ij', t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1).to(x.device)
            self.cos_cached = emb.cos()[None, None, :, :]
            self.sin_cached = emb.sin()[None, None, :, :]
        return self.cos_cached, self.sin_cached

def rotate_half(x):
    """Rotate half the hidden dims of the input"""
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    """Apply rotary position embeddings to q and k tensors
    Args:
        q, k: Query and key tensors
        cos, sin: Cached cosine and sine embeddings
    Returns:
        q and k with rotary position embeddings applied
    """
    # [batch, head, seq_len, head_dim]
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

def build_alibi_tensor(batch_size, n_heads, seq_len, device):
    """Build ALiBi (Attention with Linear Biases) tensor
    Args:
        batch_size: Batch size
        n_heads: Number of attention heads
        seq_len: Sequence length
        device: Device to create tensor on
    Returns:
        ALiBi bias tensor
    """
    def get_slopes(n_heads):
        # Function to get slopes for ALiBi
        def get_slopes_power_of_2(n_heads):
            start = 2**(-(2**-(math.log2(n_heads)-3)))
            ratio = start
            return [start * ratio**i for i in range(n_heads)]
            
        if math.log2(n_heads).is_integer():
            return get_slopes_power_of_2(n_heads)
        else:
            closest_power_of_2 = 2**math.floor(math.log2(n_heads))
            return get_slopes_power_of_2(closest_power_of_2) + get_slopes(n_heads - closest_power_of_2)
    
    slopes = torch.tensor(get_slopes(n_heads), device=device).unsqueeze(1).unsqueeze(1)
    # Create position bias
    arange_tensor = torch.arange(seq_len, device=device).unsqueeze(0).unsqueeze(0)
    alibi = slopes * arange_tensor
    
    # Expand to batch size
    return alibi.expand(batch_size, -1, -1, -1)

class SwiGLU(nn.Module):
    """SwiGLU Activation (Swish + GLU) for better performance than ReLU-based FFNs"""
    def __init__(self, hidden_dim, expansion_factor=4, dropout_prob=0.1):
        super().__init__()
        self.expanded_dim = expansion_factor * hidden_dim
        self.fc_in = nn.Linear(hidden_dim, 2 * self.expanded_dim)
        self.fc_out = nn.Linear(self.expanded_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout_prob)
    
    def forward(self, x):
        """Apply SwiGLU activation
        Args:
            x: Input tensor [batch_size, seq_len, hidden_dim]
        Returns:
            Output tensor of the same shape
        """
        x_proj = self.fc_in(x)
        x1, x2 = x_proj.chunk(2, dim=-1)
        hidden = F.silu(x1) * x2  # SwiGLU activation
        output = self.fc_out(hidden)
        return self.dropout(output)

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention with support for flash attention and ALiBi"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        assert self.n_embd % self.n_head == 0, "Embedding dimension must be divisible by number of heads"
        
        self.head_dim = self.n_embd // self.n_head
        self.q_proj = nn.Linear(self.n_embd, self.n_embd)
        self.k_proj = nn.Linear(self.n_embd, self.n_embd)
        self.v_proj = nn.Linear(self.n_embd, self.n_embd)
        self.out_proj = nn.Linear(self.n_embd, self.n_embd)
        self.dropout = nn.Dropout(config.dropout_prob)
        
        # Register causal mask for attention
        self.register_buffer(
            "causal_mask", 
            torch.tril(torch.ones(config.max_seq_len, config.max_seq_len))
        )
        
        # Initialize rotary embeddings if using them instead of ALiBi
        # (keeping both for flexibility, but typically would choose one)
        if not config.use_alibi:
            self.rotary_emb = RotaryEmbedding(self.head_dim)
    
    def forward(self, x, attention_mask=None, alibi_bias=None):
        """Forward pass for multi-head attention
        Args:
            x: Input tensor [batch_size, seq_len, n_embd]
            attention_mask: Optional mask tensor [batch_size, seq_len]
            alibi_bias: Optional ALiBi bias tensor
        Returns:
            Output tensor after attention [batch_size, seq_len, n_embd]
        """
        batch_size, seq_len, _ = x.size()
        
        # Project queries, keys, values
        q = self.q_proj(x)  # [batch_size, seq_len, n_embd]
        k = self.k_proj(x)  # [batch_size, seq_len, n_embd]
        v = self.v_proj(x)  # [batch_size, seq_len, n_embd]
        
        # Reshape for multi-head attention
        q = q.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)  # [batch_size, n_head, seq_len, head_dim]
        k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)  # [batch_size, n_head, seq_len, head_dim]
        v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)  # [batch_size, n_head, seq_len, head_dim]
        
        # Apply rotary embeddings if not using ALiBi
        if not self.config.use_alibi:
            cos, sin = self.rotary_emb(q, seq_len)
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        # Flash Attention path (preferred when available)
        if self.config.use_flash_attention and alibi_bias is None and hasattr(F, 'scaled_dot_product_attention'):
            with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=True, enable_mem_efficient=True):
                attn_output = F.scaled_dot_product_attention(
                    q, k, v, 
                    attn_mask=None,  # Handled by is_causal
                    dropout_p=self.dropout.p,
                    is_causal=True
                )
        # Standard attention path
        else:
            # Calculate attention scores
            attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # [batch_size, n_head, seq_len, seq_len]
            
            # Apply ALiBi bias if provided
            if alibi_bias is not None and self.config.use_alibi:
                attn_scores = attn_scores + alibi_bias
            
            # Apply causal mask
            mask = self.causal_mask[:seq_len, :seq_len].bool()
            attn_scores.masked_fill_(~mask.unsqueeze(0).unsqueeze(0), float('-inf'))
            
            # Apply attention mask if provided
            if attention_mask is not None:
                # Add dimensions to make it broadcastable
                attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)  # [batch_size, 1, 1, seq_len]
                attn_scores.masked_fill_(attention_mask == 0, float('-inf'))
            
            # Calculate attention weights
            attn_weights = F.softmax(attn_scores, dim=-1)
            attn_weights = self.dropout(attn_weights)
            
            # Apply attention weights to values
            attn_output = torch.matmul(attn_weights, v)  # [batch_size, n_head, seq_len, head_dim]
        
        # Reshape and project back to n_embd
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_embd)
        attn_output = self.out_proj(attn_output)
        
        return self.dropout(attn_output)

class TransformerBlock(nn.Module):
    """Transformer block with pre-norm architecture"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.attn_norm = RMSNorm(config.n_embd)
        self.ffn_norm = RMSNorm(config.n_embd)
        self.attn = MultiHeadAttention(config)
        self.ffn = SwiGLU(config.n_embd, dropout_prob=config.dropout_prob)
        self.dropout = nn.Dropout(config.dropout_prob)
    
    def forward(self, x, attention_mask=None, alibi_bias=None):
        """Forward pass for transformer block
        Args:
            x: Input tensor [batch_size, seq_len, n_embd]
            attention_mask: Optional mask tensor [batch_size, seq_len]
            alibi_bias: Optional ALiBi bias tensor
        Returns:
            Output tensor after transformer block [batch_size, seq_len, n_embd]
        """
        # Pre-normalization for attention
        normed_x = self.attn_norm(x)
        
        # Apply attention (residual connection)
        attn_output = self.attn(normed_x, attention_mask=attention_mask, alibi_bias=alibi_bias)
        x = x + attn_output
        
        # Pre-normalization for feed-forward
        normed_x = self.ffn_norm(x)
        
        # Apply feed-forward (residual connection)
        ffn_output = self.ffn(normed_x)
        x = x + ffn_output
        
        return x

class MultiheadLatentAttention(nn.Module):
    """Multi-head Latent Attention (MLA) for efficient global context aggregation"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.n_embd = config.n_embd
        self.n_head = config.n_head
        self.n_latent = config.mla_n_latent
        self.head_dim = self.n_embd // self.n_head
        
        # Latent tokens (learned)
        self.latent = nn.Parameter(torch.randn(self.n_latent, self.n_embd))
        
        # Projection matrices
        self.q_proj = nn.Linear(self.n_embd, self.n_embd)
        self.k_proj = nn.Linear(self.n_embd, self.n_embd)
        self.v_proj = nn.Linear(self.n_embd, self.n_embd)
        self.out_proj = nn.Linear(self.n_embd, self.n_embd)
        
        self.dropout = nn.Dropout(config.dropout_prob)
    
    def forward(self, x):
        """Forward pass for MLA
        Args:
            x: Input tensor [batch_size, seq_len, n_embd]
        Returns:
            Output tensor with enhanced global context [batch_size, seq_len, n_embd]
        """
        batch_size, seq_len, _ = x.size()
        
        # Expand latent tokens to batch size
        latent = self.latent.unsqueeze(0).expand(batch_size, -1, -1)  # [batch_size, n_latent, n_embd]
        
        # Project queries from latent tokens
        q = self.q_proj(latent)  # [batch_size, n_latent, n_embd]
        
        # Project keys and values from input sequence
        k = self.k_proj(x)  # [batch_size, seq_len, n_embd]
        v = self.v_proj(x)  # [batch_size, seq_len, n_embd]
        
        # Reshape for multi-head attention
        q = q.view(batch_size, self.n_latent, self.n_head, self.head_dim).transpose(1, 2)  # [batch_size, n_head, n_latent, head_dim]
        k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)  # [batch_size, n_head, seq_len, head_dim]
        v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)  # [batch_size, n_head, seq_len, head_dim]
        
        # Calculate attention scores
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # [batch_size, n_head, n_latent, seq_len]
        
        # Calculate attention weights
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention weights to values
        latent_out = torch.matmul(attn_weights, v)  # [batch_size, n_head, n_latent, head_dim]
        
        # Reshape latent output
        latent_out = latent_out.transpose(1, 2).contiguous().view(batch_size, self.n_latent, self.n_embd)  # [batch_size, n_latent, n_embd]
        
        # Project and aggregate latent information
        latent_out = self.out_proj(latent_out)  # [batch_size, n_latent, n_embd]
        
        # Simple mean aggregation of latent tokens
        aggregated = latent_out.mean(dim=1, keepdim=True)  # [batch_size, 1, n_embd]
        
        # Broadcast the aggregated latent context to the original sequence
        enhanced = x + aggregated.expand(-1, seq_len, -1)
        
        return enhanced

class GPTModel(nn.Module):
    """Core GPT model (without the language modeling head)"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Word embeddings
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        
        # Optional positional embeddings (only used if not using rotary or ALiBi)
        if not config.use_alibi:
            self.wpe = nn.Embedding(config.max_seq_len, config.n_embd)
        
        self.dropout = nn.Dropout(config.dropout_prob)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        
        # Final layer normalization
        self.ln_f = RMSNorm(config.n_embd)
        
        # Optional MLA for global context
        if config.use_mla:
            self.mla = MultiheadLatentAttention(config)
        
        # Apply weight initialization
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Initialize weights using a method similar to GPT-2"""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, input_ids, attention_mask=None):
        """Forward pass for the GPT model
        Args:
            input_ids: Input token IDs [batch_size, seq_len]
            attention_mask: Optional mask tensor [batch_size, seq_len]
        Returns:
            Output tensor after processing through the transformer [batch_size, seq_len, n_embd]
        """
        batch_size, seq_len = input_ids.shape
        assert seq_len <= self.config.max_seq_len, f"Input sequence length ({seq_len}) exceeds maximum allowed ({self.config.max_seq_len})"
        
        # Get token embeddings
        token_embeddings = self.wte(input_ids)  # [batch_size, seq_len, n_embd]
        
        # Add positional embeddings if not using ALiBi or rotary
        if not self.config.use_alibi and hasattr(self, 'wpe'):
            positions = torch.arange(0, seq_len, dtype=torch.long, device=input_ids.device)
            position_embeddings = self.wpe(positions)  # [seq_len, n_embd]
            x = token_embeddings + position_embeddings
        else:
            x = token_embeddings
        
        x = self.dropout(x)
        
        # Prepare ALiBi bias if using it
        alibi_bias = None
        if self.config.use_alibi:
            alibi_bias = build_alibi_tensor(batch_size, self.config.n_head, seq_len, x.device)
        
        # Process through transformer blocks
        for block in self.blocks:
            if self.config.use_gradient_checkpointing and self.training:
                # Apply gradient checkpointing to save memory during training
                x = checkpoint(block, x, attention_mask, alibi_bias)
            else:
                x = block(x, attention_mask=attention_mask, alibi_bias=alibi_bias)
        
        # Apply final layer normalization
        x = self.ln_f(x)
        
        # Apply MLA if enabled (for global context enhancement)
        if self.config.use_mla and hasattr(self, 'mla'):
            x = self.mla(x)
        
        return x

class GPTLMHeadModel(nn.Module):
    """GPT model with language modeling head"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = GPTModel(config)
        
        # Language modeling head (weight tied with word embeddings)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight  # Weight tying
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        """Forward pass for the GPT language model
        Args:
            input_ids: Input token IDs [batch_size, seq_len]
            attention_mask: Optional mask tensor [batch_size, seq_len]
            labels: Optional target token IDs for language modeling [batch_size, seq_len]
        Returns:
            Dictionary with loss (if labels provided) and logits
        """
        # Get transformer outputs
        transformer_outputs = self.transformer(input_ids, attention_mask=attention_mask)
        hidden_states = transformer_outputs
        
        # Get logits for next token prediction
        lm_logits = self.lm_head(hidden_states)
        
        # Calculate loss if labels are provided
        loss = None
        if labels is not None:
            # Shift logits and labels for next token prediction
            shift_logits = lm_logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            
            # Calculate cross entropy loss
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        
        return {"loss": loss, "logits": lm_logits}
    
    def generate(self, input_ids, max_length=100, temperature=1.0, top_k=50, top_p=0.9):
        """Generate text using the model
        Args:
            input_ids: Input token IDs [batch_size, seq_len]
            max_length: Maximum length of the generated sequence
            temperature: Temperature for sampling (higher = more random)
            top_k: Number of highest probability tokens to keep for top-k sampling
            top_p: Probability threshold for nucleus sampling
        Returns:
            Generated token IDs [batch_size, seq_len]
        """
        batch_size = input_ids.shape[0]
        generated = input_ids.clone()
        
        # Set model to evaluation mode
        self.eval()
        
        with torch.no_grad():
            for _ in range(max_length - input_ids.size(1)):
                outputs = self(generated)
                next_token_logits = outputs["logits"][:, -1, :] / temperature
                
                # Apply top-k filtering
                if top_k > 0:
                    # Keep only the top-k tokens
                    indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
                    next_token_logits[indices_to_remove] = float('-inf')
                
                # Apply top-p (nucleus) filtering
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
                    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    
                    # Remove tokens with cumulative probability above the threshold
                    sorted_indices_to_remove = cumulative_probs > top_p
                    # Shift the indices to the right to keep also the first token above the threshold
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    
                    for i in range(batch_size):
                        indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
                        next_token_logits[i, indices_to_remove] = float('-inf')
                
                # Sample from the filtered distribution
                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                
                # Append the sampled token to the sequence
                generated = torch.cat((generated, next_token), dim=1)
                
                # Check if all sequences have generated an EOS token
                if (next_token == self.config.eos_token_id).all():
                    break
        
        return generated


In [ ]:
# ========================================
# Data Processing
# ========================================

class TextDataModule:
    """Data module for text datasets with tokenization and batching"""
    def __init__(self, config, tokenizer):
        self.config = config
        self.tokenizer = tokenizer
        self.dataset = None
    
    def prepare_data(self):
        """Load and prepare the dataset"""
        if self.config.data_files:
            self.dataset = load_dataset('text', data_files=self.config.data_files)
        else:
            self.dataset = load_dataset(self.config.dataset_name, self.config.dataset_config)
    
    def process_dataset(self, split="train"):
        """Process and tokenize the dataset"""
        if self.dataset is None:
            self.prepare_data()
        
        def tokenize_function(examples):
            tokenized_inputs = self.tokenizer(
                examples['text'], 
                truncation=True, 
                max_length=self.config.max_seq_len, 
                padding='max_length',
                return_tensors='pt'
            )
            return tokenized_inputs
        
        # Tokenize the dataset
        tokenized_dataset = self.dataset[split].map(
            tokenize_function,
            batched=True,
            remove_columns=['text']
        )
        
        # Convert to torch format
        tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])
        
        return tokenized_dataset
    
    def get_dataloader(self, dataset, batch_size=None, shuffle=True, distributed=False):
        """Create a dataloader for the dataset"""
        if batch_size is None:
            batch_size = self.config.batch_size
        
        if distributed:
            sampler = DistributedSampler(dataset, shuffle=shuffle)
            dataloader = DataLoader(
                dataset,
                batch_size=batch_size,
                sampler=sampler,
                pin_memory=True,
                num_workers=4
            )
        else:
            dataloader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=shuffle,
                pin_memory=True,
                num_workers=4
            )
        
        return dataloader


In [ ]:
# ========================================
# Training
# ========================================

def load_checkpoint(model, optimizer=None, scheduler=None, filepath=None, map_location='cpu'):
    """Load model, optimizer, and scheduler from checkpoint
    Args:
        model: Model to load checkpoint into
        optimizer: Optional optimizer to load state
        scheduler: Optional scheduler to load state
        filepath: Path to checkpoint file
        map_location: Device to load checkpoint to
    Returns:
        model, optimizer, scheduler, epoch, global_step
    """
    if filepath is None:
        return model, optimizer, scheduler, 0, 0
    
    checkpoint = torch.load(filepath, map_location=map_location)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load optimizer state if provided
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if provided
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    # Return model, optimizer, scheduler, and training info
    return model, optimizer, scheduler, checkpoint.get('epoch', 0), checkpoint.get('global_step', 0)

def save_checkpoint(model, optimizer, scheduler, epoch, global_step, filepath):
    """Save model, optimizer, and scheduler to checkpoint
    Args:
        model: Model to save
        optimizer: Optimizer to save
        scheduler: Scheduler to save
        epoch: Current epoch
        global_step: Current global step
        filepath: Path to save checkpoint to
    """
    # Ensure directory exists
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    
    # Save model, optimizer, scheduler, and training info
    checkpoint = {
        'model_state_dict': model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'global_step': global_step
    }
    
    torch.save(checkpoint, filepath)

def train(
    config,
    model,
    train_dataloader,
    val_dataloader=None,
    tokenizer=None,
    optimizer=None,
    scheduler=None,
    device=None,
    rank=0,
    checkpoint_path=None
):
    """Train the GPT model
    Args:
        config: Configuration object
        model: Model to train
        train_dataloader: DataLoader for training data
        val_dataloader: Optional DataLoader for validation data
        tokenizer: Tokenizer for text generation during validation
        optimizer: Optimizer for training
        scheduler: Learning rate scheduler
        device: Device to train on
        rank: Rank of process in distributed training
        checkpoint_path: Path to load checkpoint from
    Returns:
        Trained model
    """
    # Setup optimizer and scheduler if not provided
    if optimizer is None:
        optimizer = AdamW(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )
    
    if scheduler is None:
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=config.warmup_steps,
            num_training_steps=len(train_dataloader) * config.epochs
        )
    
    # Load checkpoint if provided
    if checkpoint_path is not None:
        model, optimizer, scheduler, start_epoch, global_step = load_checkpoint(
            model, optimizer, scheduler, checkpoint_path, device
        )
    else:
        start_epoch, global_step = 0, 0
    
    # Setup for mixed precision training
    scaler = GradScaler(enabled=config.use_mixed_precision and device.type == 'cuda')
    
    # Move model to device
    model = model.to(device)
    
    # Wrap model for distributed training if needed
    if config.distributed:
        model = DDP(model, device_ids=[rank])
    
    # Training loop
    best_val_loss = float('inf')
    epochs = config.epochs
    checkpoint_interval = config.checkpoint_interval
    last_checkpoint_time = time.time()
    
    if rank == 0:
        logging.info(f"Starting training for {epochs} epochs")
    
    model.train()
    for epoch in range(start_epoch, epochs):
        # Set epoch for distributed sampler
        if config.distributed and hasattr(train_dataloader.sampler, 'set_epoch'):
            train_dataloader.sampler.set_epoch(epoch)
        
        if rank == 0:
            print(f"Epoch {epoch+1}/{epochs}")
            epoch_iterator = tqdm(train_dataloader, desc="Training")
        else:
            epoch_iterator = train_dataloader
        
        # Training
        model.train()
        for step, batch in enumerate(epoch_iterator):
            # Reset gradients
            optimizer.zero_grad()
            
            # Move batch to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            # Forward pass with mixed precision
            with autocast(enabled=config.use_mixed_precision and device.type == 'cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                loss = outputs["loss"]
                
                # Scale loss for gradient accumulation
                loss = loss / config.gradient_accumulation_steps
            
            # Backward pass with mixed precision
            scaler.scale(loss).backward()
            
            # Only update parameters and learning rate on gradient accumulation steps
            if (step + 1) % config.gradient_accumulation_steps == 0:
                # Clip gradients to prevent exploding gradients
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                # Update parameters with mixed precision
                scaler.step(optimizer)
                scaler.update()
                
                # Update learning rate
                scheduler.step()
                
                # Increment global step
                global_step += 1
            
            # Print loss
            if rank == 0 and (step + 1) % 10 == 0:
                if isinstance(epoch_iterator, tqdm):
                    epoch_iterator.set_postfix(loss=loss.item() * config.gradient_accumulation_steps)
                else:
                    print(f"Step {step+1}, Loss: {loss.item() * config.gradient_accumulation_steps}")
            
            # Save checkpoint
            if rank == 0 and time.time() - last_checkpoint_time >= checkpoint_interval:
                checkpoint_name = f"checkpoint-epoch{epoch+1}-step{global_step}.pt"
                checkpoint_path = os.path.join(config.checkpoint_dir, checkpoint_name)
                save_checkpoint(model, optimizer, scheduler, epoch, global_step, checkpoint_path)
                
                if rank == 0:
                    logging.info(f"Saved checkpoint at step {global_step}")
                
                last_checkpoint_time = time.time()
        
        # Validation
        if val_dataloader is not None and rank == 0:
            model.eval()
            val_losses = []
            
            with torch.no_grad():
                val_iterator = tqdm(val_dataloader, desc="Validation")
                
                for batch in val_iterator:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    
                    with autocast(enabled=config.use_mixed_precision and device.type == 'cuda'):
                        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                        loss = outputs["loss"]
                    
                    val_losses.append(loss.item())
            
            # Calculate average validation loss
            avg_val_loss = sum(val_losses) / len(val_losses)
            
            # Log validation loss
            if rank == 0:
                logging.info(f"Epoch {epoch+1} - Validation Loss: {avg_val_loss}")
                print(f"Validation Loss: {avg_val_loss}")
            
            # Generate sample text for validation
            if tokenizer is not None and rank == 0:
                model.eval()
                
                # Get sample batch
                sample_batch = next(iter(val_dataloader))
                sample_ids = sample_batch['input_ids'][0:1].to(device)
                
                # Generate text continuation
                generated_ids = model.module.generate(sample_ids) if hasattr(model, 'module') else model.generate(sample_ids)
                
                # Decode generated text
                sample_text = tokenizer.decode(sample_ids[0], skip_special_tokens=True)
                generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                
                # Log sample generation
                if rank == 0:
                    logging.info(f"Sample prompt: {sample_text[:50]}...")
                    logging.info(f"Sample generation: {generated_text}")
                    print(f"Sample prompt: {sample_text[:50]}...")
                    print(f"Generated: {generated_text}")
            
            # Save best model
            if avg_val_loss < best_val_loss and rank == 0:
                best_val_loss = avg_val_loss
                best_model_path = os.path.join(config.checkpoint_dir, "best_model.pt")
                save_checkpoint(model, optimizer, scheduler, epoch, global_step, best_model_path)
                
                if rank == 0:
                    logging.info(f"New best model saved with validation loss: {best_val_loss}")
            
            # Set model back to training mode
            model.train()
        
        # Save checkpoint at the end of each epoch
        if rank == 0:
            checkpoint_name = f"checkpoint-epoch{epoch+1}.pt"
            checkpoint_path = os.path.join(config.checkpoint_dir, checkpoint_name)
            save_checkpoint(model, optimizer, scheduler, epoch, global_step, checkpoint_path)
            
            if rank == 0:
                logging.info(f"Epoch {epoch+1} completed. Checkpoint saved.")
    
    # Save final model
    if rank == 0:
        final_model_path = os.path.join(config.checkpoint_dir, "final_model.pt")
        save_checkpoint(model, optimizer, scheduler, epochs-1, global_step, final_model_path)
        
        if rank == 0:
            logging.info("Training completed. Final model saved.")
    
    return model

def train_distributed(config, rank, world_size):
    """Train model in distributed mode
    Args:
        config: Configuration object
        rank: Rank of process
        world_size: Total number of processes
    """
    # Setup distributed training
    setup_distributed(rank, world_size)
    
    # Setup environment
    device = setup_environment(config, rank)
    
    # Initialize tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    
    # Initialize model
    model = GPTLMHeadModel(config)
    
    # Setup data
    data_module = TextDataModule(config, tokenizer)
    train_dataset = data_module.process_dataset(split="train")
    val_dataset = data_module.process_dataset(split="validation") if "validation" in data_module.dataset else None
    
    train_dataloader = data_module.get_dataloader(train_dataset, distributed=True)
    val_dataloader = data_module.get_dataloader(val_dataset, distributed=False) if val_dataset else None
    
    # Setup optimizer and scheduler
    optimizer = AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config.warmup_steps,
        num_training_steps=len(train_dataloader) * config.epochs
    )
    
    # Find latest checkpoint
    checkpoint_path = None
    if os.path.exists(config.checkpoint_dir):
        checkpoints = [os.path.join(config.checkpoint_dir, f) for f in os.listdir(config.checkpoint_dir) if f.startswith("checkpoint-epoch") and f.endswith(".pt")]
        if checkpoints:
            checkpoint_path = max(checkpoints, key=os.path.getctime)
    
    # Train model
    train(
        config=config,
        model=model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        tokenizer=tokenizer,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        rank=rank,
        checkpoint_path=checkpoint_path
    )
    
    # Cleanup distributed training
    dist.destroy_process_group()


In [ ]:
# ========================================
# Inference
# ========================================

def load_model_for_inference(config, checkpoint_path, device='cpu'):
    """Load model for inference
    Args:
        config: Configuration object
        checkpoint_path: Path to model checkpoint
        device: Device to load model on
    Returns:
        Loaded model
    """
    # Initialize model
    model = GPTLMHeadModel(config)
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Move model to device
    model = model.to(device)
    
    # Set model to evaluation mode
    model.eval()
    
    return model

def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.7, top_k=50, top_p=0.9, device='cpu'):
    """Generate text using the model
    Args:
        model: Model to use for generation
        tokenizer: Tokenizer for encoding/decoding text
        prompt: Prompt to start generation from
        max_length: Maximum length of the generated sequence
        temperature: Temperature for sampling (higher = more random)
        top_k: Number of highest probability tokens to keep for top-k sampling
        top_p: Probability threshold for nucleus sampling
        device: Device to use for generation
    Returns:
        Generated text
    """
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # Generate continuation
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )
    
    # Decode output
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    return generated_text


In [ ]:
# ========================================
# Main Training Function
# ========================================

def main():
    """Main function to run the GPT training pipeline"""
    # Define configuration
    config = Config(
        # Model
        vocab_size=50257,  # GPT-2 vocabulary size
        max_seq_len=1024,
        n_embd=768,  # 768 for GPT-2 small
        n_layer=12,
        n_head=12,
        dropout_prob=0.1,
        
        # Training
        batch_size=16,
        gradient_accumulation_steps=2,
        epochs=5,
        warmup_steps=1000,
        learning_rate=5e-5,
        weight_decay=0.01,
        
        # Advanced techniques
        use_alibi=True,
        use_flash_attention=True,
        use_gradient_checkpointing=True,
        use_mixed_precision=True,
        use_mla=True,
        mla_n_latent=16,
        
        # Distributed
        distributed=torch.cuda.device_count() > 1,
        world_size=torch.cuda.device_count() if torch.cuda.is_available() else 1,
        
        # Data
        dataset_name="wikitext",
        dataset_config="wikitext-103-v1",
        
        # Checkpointing
        checkpoint_dir="./checkpoints",
        checkpoint_interval=600  # 10 minutes
    )
    
    # Set up logging and checkpointing directories
    os.makedirs('logs', exist_ok=True)
    os.makedirs(config.checkpoint_dir, exist_ok=True)
    
    # Run distributed training if multiple GPUs are available
    if config.distributed:
        mp.spawn(
            train_distributed,
            args=(config, config.world_size),
            nprocs=config.world_size,
            join=True
        )
    # Otherwise run single-GPU or CPU training
    else:
        # Set up environment
        device = setup_environment(config)
        
        # Initialize tokenizer
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token
        
        # Initialize model
        model = GPTLMHeadModel(config)
        
        # Setup data
        data_module = TextDataModule(config, tokenizer)
        train_dataset = data_module.process_dataset(split="train")
        val_dataset = data_module.process_dataset(split="validation") if "validation" in data_module.dataset else None
        
        train_dataloader = data_module.get_dataloader(train_dataset)
        val_dataloader = data_module.get_dataloader(val_dataset) if val_dataset else None
        
        # Find latest checkpoint
        checkpoint_path = None
        if os.path.exists(config.checkpoint_dir):
            checkpoints = [os.path.join(config.checkpoint_dir, f) for f in os.listdir(config.checkpoint_dir) if f.startswith("checkpoint-epoch") and f.endswith(".pt")]
            if checkpoints:
                checkpoint_path = max(checkpoints, key=os.path.getctime)
        
        # Train model
        train(
            config=config,
            model=model,
            train_dataloader=train_dataloader,
            val_dataloader=val_dataloader,
            tokenizer=tokenizer,
            device=device,
            checkpoint_path=checkpoint_path
        )

# ========================================
# Example Usage
# ========================================

if __name__ == "__main__":
    # Run training
    main()
    
    # Example inference code:
    """
    # Load configuration (adjust as needed)
    config = Config(
        vocab_size=50257,
        max_seq_len=1024,
        n_embd=768,
        n_layer=12,
        n_head=12,
        dropout_prob=0.1,
        use_alibi=True,
        use_mla=True,
        mla_n_latent=16,
        checkpoint_dir="./checkpoints"
    )
    
    # Load tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load model (use best or final model)
    model_path = os.path.join(config.checkpoint_dir, "best_model.pt")
    if not os.path.exists(model_path):
        model_path = os.path.join(config.checkpoint_dir, "final_model.pt")
    
    model = load_model_for_inference(config, model_path, device)
    
    # Generate text from a prompt
    prompt = "Once upon a time, in a land far away,"
    generated_text = generate_text(model, tokenizer, prompt, max_length=200, device=device)
    print(f"Generated text:\n{generated_text}")
    """


# Example Usage

Below is an example of how to load the configuration, tokenizer, model and generate text (this code is commented out in the main cell):

```python
# Load configuration (adjust as needed)
config = Config(
    vocab_size=50257,
    max_seq_len=1024,
    n_embd=768,
    n_layer=12,
    n_head=12,
    dropout_prob=0.1,
    use_alibi=True,
    use_mla=True,
    mla_n_latent=16,
    checkpoint_dir="./checkpoints"
)

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model (use best or final model)
model_path = os.path.join(config.checkpoint_dir, "best_model.pt")
if not os.path.exists(model_path):
    model_path = os.path.join(config.checkpoint_dir, "final_model.pt")

model = load_model_for_inference(config, model_path, device)

# Generate text from a prompt
prompt = "Once upon a time, in a land far away,"
generated_text = generate_text(model, tokenizer, prompt, max_length=200, device=device)
print(f"Generated text:\n{generated_text}")
```
